##### NOTEBOOK 7 — Validacion Silver
##### TFM: Integración QML en pipeline DataOps
##### Juan Albornoz Carrasco — Universidad Europea de Valencia
##### Extract desde AWS S3, Load en Delta Lake

##### CELDA 1 — Instalación dataframe-expectations

In [0]:
#%pip install dataframe-expectations==0.7.0 --quiet

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


##### CELDA 2 — Imports

In [0]:
import os
import time
import pandas as pd
from dataframe_expectations import (
    DataFrameExpectationsSuite,
    DataFrameExpectationsSuiteRunner
)
print("Imports OK")

Imports OK


##### CELDA 3 — Reinicio obligatorio post-instalación

In [0]:
#dbutils.library.restartPython()

##### CELDA 4 — Validación calidad dato capa Silver
##### dataframe-expectations 0.7.0
##### Notebook independiente — no modifica notebook_02_silver

In [0]:
# Ruta Silver Parquet
silver_path = "/Volumes/workspace/default/nhanes/silver/silver_svm.parquet"

# Cargar dataset Silver
print("Cargando dataset Silver...")
df = pd.read_parquet(silver_path)

print(f"Registros: {df.shape[0]}, Columnas: {df.shape[1]}")
print(f"TARGET: {df['TARGET'].value_counts().to_dict()}\n")

# Definir suite de validaciones
suite = DataFrameExpectationsSuite(suite_name="silver_quality_suite")

# DIMENSION 1: Completitud
suite.expect_max_null_count(column_name="TARGET",   max_count=0)
suite.expect_max_null_count(column_name="LBXGH",    max_count=0)
suite.expect_max_null_count(column_name="LBXGLU",   max_count=0)
suite.expect_max_null_count(column_name="RIDAGEYR",  max_count=0)
suite.expect_max_null_count(column_name="BMXBMI",    max_count=0)

# DIMENSION 2: Rangos clinicos validos
suite.expect_column_min_between(column_name="RIDAGEYR", min_value=18,   max_value=25)
suite.expect_column_max_between(column_name="RIDAGEYR", min_value=70,   max_value=120)
suite.expect_column_min_between(column_name="LBXGH",    min_value=3.0,  max_value=6.0)
suite.expect_column_max_between(column_name="LBXGH",    min_value=8.0,  max_value=20.0)
suite.expect_column_min_between(column_name="LBXGLU",   min_value=30,   max_value=80)
suite.expect_column_max_between(column_name="LBXGLU",   min_value=150,  max_value=500)
suite.expect_column_min_between(column_name="BMXBMI",   min_value=10.0, max_value=18)
suite.expect_column_max_between(column_name="BMXBMI",   min_value=40.0, max_value=80)

# DIMENSION 3: Volumen esperado
suite.expect_min_rows(min_rows=7000)
suite.expect_max_rows(max_rows=9000)

# Construir runner y ejecutar
t0 = time.time()
runner = suite.build()
results = runner.run(df)
t1 = time.time()

Cargando dataset Silver...
Registros: 7831, Columnas: 91
TARGET: {0: 6732, 1: 1099}



##### CELDA 5 — Resumen ejecutivo para figura TFM

In [0]:
print("=== RESUMEN VALIDACION SILVER ===")
print(f"Suite:               {results.suite_name}")
print(f"Fecha:               {results.start_time.date()}")
print(f"Registros validados: {results.dataframe_row_count}")
print(f"Total expectativas:  {results.total_expectations}")
print(f"Passed:              {results.total_passed}")
print(f"Failed:              {results.total_failed}")
print(f"Pass rate:           {results.pass_rate}")
print(f"Exito:               {results.success}")
print(f"Duracion:            {results.total_duration_seconds:.4f} segundos")
print()
print("Detalle por expectativa:")
for r in results.results:
    estado = "PASSED" if r.status.value == "passed" else "FAILED"
    print(f"  [{estado}] {r.description}")

=== RESUMEN VALIDACION SILVER ===
Suite:               silver_quality_suite
Fecha:               2026-06-21
Registros validados: 7831
Total expectativas:  15
Passed:              15
Failed:              0
Pass rate:           1.0
Exito:               True
Duracion:            0.0018 segundos

Detalle por expectativa:
  [PASSED] column 'TARGET' has at most 0 null values
  [PASSED] column 'LBXGH' has at most 0 null values
  [PASSED] column 'LBXGLU' has at most 0 null values
  [PASSED] column 'RIDAGEYR' has at most 0 null values
  [PASSED] column 'BMXBMI' has at most 0 null values
  [PASSED] column 'RIDAGEYR' minimum value between 18 and 25
  [PASSED] column 'RIDAGEYR' maximum value between 70 and 120
  [PASSED] column 'LBXGH' minimum value between 3.0 and 6.0
  [PASSED] column 'LBXGH' maximum value between 8.0 and 20.0
  [PASSED] column 'LBXGLU' minimum value between 30 and 80
  [PASSED] column 'LBXGLU' maximum value between 150 and 500
  [PASSED] column 'BMXBMI' minimum value between 10

##### CELDA 6 — Persistencia resultados en Unity Catalog Volumes

In [0]:
models_dir = "/Volumes/workspace/default/nhanes/models"

resumen = {
    "suite_name":         results.suite_name,
    "fecha":              str(results.start_time.date()),
    "total_expectations": results.total_expectations,
    "total_passed":       results.total_passed,
    "total_failed":       results.total_failed,
    "pass_rate":          results.pass_rate,
    "success":            results.success,
    "dataframe_rows":     results.dataframe_row_count,
    "duration_seconds":   results.total_duration_seconds
}

pd.DataFrame([resumen]).to_csv(
    f"{models_dir}/validacion_silver_dfe.csv",
    index=False
)

print(f"Resultados guardados: {models_dir}/validacion_silver_dfe.csv")
print("\nResumen:")
for k, v in resumen.items():
    print(f"  {k}: {v}")

Resultados guardados: /Volumes/workspace/default/nhanes/models/validacion_silver_dfe.csv

Resumen:
  suite_name: silver_quality_suite
  fecha: 2026-06-21
  total_expectations: 15
  total_passed: 15
  total_failed: 0
  pass_rate: 1.0
  success: True
  dataframe_rows: 7831
  duration_seconds: 0.001751
